# BiRefNet Adapter Fine-tuning 검증 노트북

원본 파라미터 완전 freeze + 어댑터(LoRA, AdapterConv)만 학습하는 구조를 검증합니다.

1. Import 검증
2. 어댑터 적용 & 파라미터 확인
3. Forward 검증 (train / eval)
4. 원본 파라미터 freeze 검증
5. Loss 검증
6. 1 Step 학습 통합 검증
7. 어댑터 저장/로드 검증

## 0. 경로 설정

In [ ]:
import os, sys

FINETUNE_DIR = os.path.join(os.getcwd(), 'finetune') if 'finetune' not in os.getcwd() else os.getcwd()
if FINETUNE_DIR not in sys.path:
    sys.path.insert(0, FINETUNE_DIR)

print(f"Working dir: {os.getcwd()}")
print(f"Finetune dir: {FINETUNE_DIR}")

## 1. Import 검증

In [ ]:
from model import FineTuneBiRefNet
from dataset import FineTuneDataset
from loss import SegmentationLoss
from adapters import LoRALinear, AdapterConv

print('✅ 모든 모듈 import 성공')

## 2. 모델 생성 & 어댑터 확인

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = FineTuneBiRefNet(
    pretrained=False,    # 구조만 확인 (가중치 다운로드 생략)
    lora_rank=4,
    lora_scale=1.0,
    adapter_reduction=4,
).to(device)

print('\n✅ 모델 생성 성공')

In [ ]:
# 어댑터 파라미터 상세
print('=== 학습 가능한(adapter) 파라미터 목록 ===')
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {name:70s}  {str(list(p.shape)):20s}  numel={p.numel():>8,}")

adapter_params = model.get_adapter_params()
print(f"\n총 학습 가능 파라미터 수: {sum(p.numel() for p in adapter_params):,}")

## 3. Forward 검증

In [ ]:
dummy = torch.randn(1, 3, 512, 512).to(device)

# 학습 모드
model.train()
with torch.no_grad():
    pred_train = model(dummy)
print(f"✅ Forward (train): {pred_train.shape}")
print(f"   aux_loss (Gradient Ref): {model.aux_loss.item():.4f}")

# 추론 모드
model.eval()
with torch.no_grad():
    pred_eval = model(dummy)
print(f"✅ Forward (eval):  {pred_eval.shape}")

## 4. 원본 파라미터 freeze 검증

원본 BiRefNet의 파라미터가 단 하나도 `requires_grad=True`가 아닌지 확인합니다.

In [ ]:
# 학습 불가능한 파라미터 중 'lora_' 또는 'adapter'가 포함된 것은 없어야 함
# 학습 가능한 파라미터 중 'lora_' 또는 'adapter'가 없는 것은 없어야 함

leaked = []
for name, p in model.named_parameters():
    is_adapter = any(kw in name for kw in ['lora_', 'adapter'])
    if p.requires_grad and not is_adapter:
        leaked.append(name)

if leaked:
    print('❌ 원본 파라미터 중 freeze 되지 않은 것 발견!')
    for n in leaked:
        print(f'   {n}')
else:
    print('✅ 원본 파라미터 전부 freeze 확인 완료 (학습 가능 = 어댑터만)')

## 5. Loss 검증

In [ ]:
criterion = SegmentationLoss()

dummy_pred = torch.randn(2, 1, 256, 256, device=device)
dummy_gt = torch.rand(2, 1, 256, 256, device=device)

loss = criterion(dummy_pred, dummy_gt)
print(f"✅ Loss 계산: {loss.item():.4f}  (requires_grad={loss.requires_grad})")

## 6. 1 Step 학습 통합 검증

In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader
from torch.optim import AdamW

# 더미 데이터 생성
dummy_img_dir = '/tmp/birefnet_test/images'
dummy_mask_dir = '/tmp/birefnet_test/masks'
os.makedirs(dummy_img_dir, exist_ok=True)
os.makedirs(dummy_mask_dir, exist_ok=True)
for i in range(4):
    Image.fromarray(np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8)).save(f'{dummy_img_dir}/{i:03d}.png')
    Image.fromarray(np.random.randint(0, 2, (256, 256), dtype=np.uint8) * 255).save(f'{dummy_mask_dir}/{i:03d}.png')

ds = FineTuneDataset(dummy_img_dir, dummy_mask_dir, size=(512, 512))
dl = DataLoader(ds, batch_size=2, shuffle=True, drop_last=True)

# 학습 설정
model.train()
optimizer = AdamW(model.get_adapter_params(), lr=1e-4)
criterion = SegmentationLoss()

# 어댑터 파라미터 스냅샷 (학습 전)
adapter_before = {n: p.clone() for n, p in model.named_parameters() if p.requires_grad}

# 원본 파라미터 스냅샷 (학습 전)
orig_before = {n: p.clone() for n, p in model.named_parameters() if not p.requires_grad}

# 1 step
images, masks = next(iter(dl))
images, masks = images.to(device), masks.to(device)

optimizer.zero_grad()
preds = model(images)
loss_seg = criterion(preds, masks)
total_loss = loss_seg + model.aux_loss
total_loss.backward()
optimizer.step()

print(f"✅ 1 Step 학습 완료")
print(f"   loss_seg={loss_seg.item():.4f}, loss_gdt={model.aux_loss.item():.4f}, total={total_loss.item():.4f}")

# 어댑터 파라미터가 변했는지 확인
adapter_changed = sum(
    1 for n, p in model.named_parameters()
    if p.requires_grad and not torch.equal(p.data, adapter_before[n])
)
print(f"   어댑터 파라미터 변경: {adapter_changed}/{len(adapter_before)}개")

# 원본 파라미터가 안 변했는지 확인
orig_changed = sum(
    1 for n, p in model.named_parameters()
    if not p.requires_grad and not torch.equal(p.data, orig_before[n])
)
if orig_changed == 0:
    print(f"✅ 원본 파라미터 변경: 0개 (완전 보존 확인!)")
else:
    print(f"❌ 원본 파라미터 {orig_changed}개 변경됨!")

## 7. 어댑터 저장/로드 검증

In [ ]:
save_path = '/tmp/birefnet_test/test_adapter.pth'

# 저장
model.save_adapters(save_path)
file_size = os.path.getsize(save_path) / 1024 / 1024
print(f"   파일 크기: {file_size:.2f} MB")

# 새 모델에 로드
model2 = FineTuneBiRefNet(pretrained=False, lora_rank=4).to(device)
model2.load_adapters(save_path)

# 로드 후 동일 출력 확인
model.eval()
model2.eval()
with torch.no_grad():
    out1 = model(dummy[:1].to(device))
    out2 = model2(dummy[:1].to(device))

if torch.allclose(out1, out2, atol=1e-5):
    print('✅ 어댑터 저장/로드 후 동일 출력 확인!')
else:
    diff = (out1 - out2).abs().max().item()
    print(f'⚠️ 출력 차이: {diff:.6f} (pretrained 가중치 차이 가능)')

In [ ]:
# 정리
import shutil
shutil.rmtree('/tmp/birefnet_test', ignore_errors=True)

print("\n" + "="*50)
print("모든 검증 완료! 🎉")
print("="*50)